# r9p step B2: r9p on the gauntlet, consistency and unseen suites
About 2-4 h.

In [ ]:
import os, glob, json, hashlib, shutil, subprocess
W = '/kaggle/working'
PUB, PRIV = f'{W}/public', f'{W}/private'   # public: counts only, safe to commit; private: real question text, never commit
os.makedirs(PUB, exist_ok=True)
os.makedirs(PRIV, exist_ok=True)
REPO, X
R8P_SHA = '38c289d351d6517b5cde1bc9fb64ea97bd33eee193639d6351edda52fdfd7a52'


def sh(cmd):
    """Runs one shell command; stops the notebook if it fails."""
    print('$', cmd, flush=True)
    if subprocess.run(cmd, shell=True).returncode:
        raise SystemExit(f'STOP: command failed: {cmd}')


def stop(msg):
    raise SystemExit('STOP: ' + msg)


def find(pattern):
    return sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True))


def one(pattern, what):
    got = find(pattern)
    if not got:
        stop(f'attach {what} (no {pattern} under /kaggle/input)')
    return got[0]


def sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()


os.makedirs('/kaggle/temp', exist_ok=True)
if not os.path.exists(REPO):
    sh(f'git clone --depth 1 -b {BRANCH} https://github.com/ArAnirudh2901/Kinthread-llm-model.git {REPO}')
os.chdir(REPO)
tgz = find('real_questions_full.tgz')
if not tgz:
    stop('attach the private dataset kinthread-real-questions (it holds real_questions_full.tgz)')
sh(f'rm -rf data/real_questions && tar -xzf {tgz[0]}')
if not glob.glob('data/real_questions/*.jsonl'):
    stop('no real question files after unpacking real_questions_full.tgz')
print('repo and real questions ready')


def r8p_checkpoint():
    cands = find('best.pt')
    try:
        import kagglehub
        for h in ('anirudharavalli/notebook99ae0586bd/versions/7', 'anirudharavalli/notebook99ae0586bd/versions/5'):
            try:
                cands += glob.glob(f'{kagglehub.notebook_output_download(h)}/**/best.pt', recursive=True)
            except Exception as e:
                print(h, e)
    except Exception as e:
        print('kagglehub:', e)
    for p in cands:
        if sha(p) == R8P_SHA:
            return p
    stop(f'no best.pt with SHA-256 {R8P_SHA} among {cands}')


os.makedirs('runs/v10-r8p-final', exist_ok=True)
if not os.path.exists('runs/v10-r8p-final/best.pt'):
    shutil.copy(r8p_checkpoint(), 'runs/v10-r8p-final/best.pt')
print('r8p checkpoint ready', sha('runs/v10-r8p-final/best.pt')[:12])

finals = [p for p in find('FINAL.json') if os.path.exists(os.path.join(os.path.dirname(p), 'verifier.json'))]
if not finals:
    stop('attach the final-selection notebook output (notebookf8798397a0: FINAL.json and verifier.json)')
shutil.copy(finals[0], 'runs/v10-r8p-final/FINAL.json')
shutil.copy(os.path.join(os.path.dirname(finals[0]), 'verifier.json'), 'runs/v10-r8p-final/verifier.json')
R8P_FINAL = '--adaptive 1.0 --hybrid --verifier runs/v10-r8p-final/verifier.json --cover --split_search --para 2 --k 16'
print('r8p FINAL setting:', R8P_FINAL)

MODEL = 'r9p'
CKPT = one('runs/v10-r9p/best.pt', 'the output of kinthread-r9p-c-train')
print(MODEL, CKPT, sha(CKPT)[:12])

In [ ]:
sh(f'python -m eval.gauntlet --ckpt {CKPT} --k 16 --out {PUB}/{MODEL}_gauntlet.json')
sh(f'python -m eval.consistency --groups 300 --ckpt {CKPT} --k 16 --out {PUB}/{MODEL}_consistency.json')
sh(f'python -m eval.evaluate --ckpt {CKPT} --sets unseen --out {PUB}/{MODEL}_unseen.json')

In [ ]:
g = json.load(open(f'{PUB}/{MODEL}_gauntlet.json'))
c = json.load(open(f'{PUB}/{MODEL}_consistency.json'))
summary = {'step': f'B-{MODEL}', 'checkpoint_sha256': sha(CKPT), 'gauntlet_correct': g['correct'],
           'gauntlet_by_category': {k: v['correct'] for k, v in g['by category'].items()},
           'consistency_all_agree': c['all variants agree'], 'consistency_by_transform': c['by transform']}
json.dump(summary, open(f'{PUB}/B_{MODEL}_summary.json', 'w'), indent=1)
print(json.dumps(summary, indent=1))
print('B DONE')